# Logistic Regression

Logistic regression is mainly used for binary classification problems. Unlike linear regression which predicts a continuous value, the output here is a probability between 0 and 1, indicating the likelihood of belonging to a class.

---

## The Model

The formula is similar to linear regression with one key difference, the output is passed through a **sigmoid function** that squashes it between 0 and 1:

$$f_{\mathbf{w},b}(\mathbf{x}) = g(\mathbf{w} \cdot \mathbf{x} + b)$$

Where each variable represents:

- $f_{\mathbf{w},b}(\mathbf{x})$: The predicted probability
- $\mathbf{w}$: The weight vector, one weight per input feature
- $b$: The bias
- $\mathbf{x}$: The input feature vector
- $g$: The sigmoid function

---

## The Sigmoid Function

$$g(z) = \frac{1}{1 + e^{-z}}$$

Substituting $z = \mathbf{w} \cdot \mathbf{x} + b$ gives us the full model:

$$f_{\mathbf{w},b}(\mathbf{x}) = \frac{1}{1 + e^{-(\mathbf{w} \cdot \mathbf{x} + b)}}$$

---

## The Cost Function

The cost function (which is used to measure how wrong our model is) is calculated using binary cross-entropy. A key property of this loss function is:

- **Asymmetric penalties:** The model is penalized heavily when it is confidently wrong (e.g. predicting $0.05$ when the true value is $1$)

$$J(\mathbf{w},b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(f_{\mathbf{w},b}(\mathbf{x}^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\mathbf{w},b}(\mathbf{x}^{(i)})) \right]$$

Where:

- $J(\mathbf{w},b)$: The cost
- $m$: The number of training examples
- $i$: Denotes which example we are on
- $\mathbf{x}^{(i)}$: The $i$ th input feature vector
- $y^{(i)}$: The $i$ th actual output value (either 0 or 1)
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$: The predicted probability for the $i$ th example

---

## Gradient Descent

Just like linear regression, we minimize the cost using gradient descent. The update rules are identical in form:

$$\mathbf{w} := \mathbf{w} - \alpha \frac{\partial J(\mathbf{w},b)}{\partial \mathbf{w}}$$

$$b := b - \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}$$

Where the partial derivatives expand to:

$$\frac{\partial J(\mathbf{w},b)}{\partial \mathbf{w}} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \mathbf{x}^{(i)}$$

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$

The gradient equations look identical to those in linear regression. The difference is that $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the sigmoid output, not a raw linear value.

In [8]:
import numpy as np
import plotly.express as px
z = np.linspace(-10,10,100)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

f = px.line(x=z, y=sigmoid(z), title="logistic function")
f.show()    
    

In [21]:
def standardize_x(x_new):
    return (x_new - np.mean(x_train)) / np.std(x_train)

def standardize_y(y_new):
    return (y_new - np.mean(y_train)) / np.std(y_train)


class LogisticRegression:
    
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
        self.w = None
        self.b = 0.0
    
    def predict(self, X):
        z = np.dot(X, self.w) + self.b
        return sigmoid(z)
    
    def loss(self, x_train, y_train):
        m = len(x_train)
        return -(1/m)* (np.sum( (y_train*(np.log(self.predict(x_train))) + ((1-y_train)*(np.log(1-self.predict(x_train)))))))
    
    def gradient_descent(self, x_train, y_train):
        m = len(x_train)
        weight_partial = (1/m) * np.dot(x_train.T, (self.predict(x_train) - y_train))
        
        bias_partial = (1/m) * np.sum(self.predict(x_train)-y_train)
        
        self.w -= self.learning_rate * weight_partial
        self.b -= self.learning_rate * bias_partial
    
    def fit(self, x_train, y_train, epochs):
        self.w = np.zeros(x_train.shape[1])
        losses = []
        
        for i in range(epochs):
            loss = self.loss(x_train, y_train)
            self.gradient_descent(x_train, y_train)
            losses.append(loss)
        
        fig = px.line(y=losses, title="Loss vs Iteration", template="plotly_dark")
        fig.update_layout(
            title_font_color="#41BEE9",
            xaxis=dict(color="#41BEE9", title="Iterations"),
            yaxis=dict(color="#41BEE9", title="Loss")
        )

        fig.show()

model = LogisticRegression(.001)

np.random.seed(42)
n = 200

hours_studied = np.random.uniform(1, 10, n)
prev_score = np.random.uniform(40, 100, n)

score = 0.5 * hours_studied + 0.05 * prev_score + np.random.normal(0, 0.5, n)
passed = (score > np.percentile(score, 50)).astype(int)

data = {
    'HoursStudied': hours_studied,
    'PrevScore': prev_score,
    'Passed': passed
}

x_train = np.column_stack([hours_studied, prev_score])
y_train = passed

fig = px.scatter(
    x=hours_studied, 
    y=prev_score, 
    color=passed.astype(str),
    title="Pass/Fail by Hours Studied and Previous Score",
    labels={'x': 'Hours Studied', 'y': 'Previous Score', 'color': 'Passed'}
)
fig.show()

model.fit(x_train, y_train, 3000)

